In [2]:
# ===== CELL 1: Connect Google Drive =====
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# ===== CELL 2: Create the project folders and the master config =====
import os, json, datetime

BASE = os.environ.get("BETH_BASE", "/content/drive/MyDrive/BETH_Triage")

FOLDERS = [
    "00_raw_zips",        # the 3 zip files you upload (never edited)
    "01_data_raw",        # unzipped CSVs (read-only copies)
    "02_data_processed",  # cleaned feature files made later
    "03_models",          # saved trained models
    "04_scores",          # anomaly scores per record
    "05_results",         # metric tables (CSV)
    "06_figures",         # charts (PNG/PDF)
    "07_logs",            # hashes, versions, run logs
    "08_triage_output",   # Forensic_Triage_Queue.csv
]
for f in FOLDERS:
    os.makedirs(os.path.join(BASE, f), exist_ok=True)

CONFIG_PATH = os.path.join(BASE, "config.json")
if not os.path.exists(CONFIG_PATH):
    config = {
        "project": "Unsupervised Forensic Evidence Triage (Member 3, J26-CS-308)",
        "created": datetime.datetime.now().isoformat(timespec="seconds"),
        "random_seed": 42,
        "dataset": {
            "name": "BETH (BPF-extended Tracking Honeypot), labelled process-log subset",
            "source_url": "https://www.kaggle.com/datasets/katehighnam/beth-dataset",
            "paper": "Highnam et al., BETH Dataset: Real Cybersecurity Data for Anomaly Detection Research, ICML 2021 UDL Workshop",
            "files": ["labelled_training_data.csv",
                      "labelled_validation_data.csv",
                      "labelled_testing_data.csv"],
        },
        "decisions": [],
    }
    with open(CONFIG_PATH, "w") as fh:
        json.dump(config, fh, indent=2)
    print("Created new config.json")
else:
    print("config.json already exists - kept it unchanged")

print("\nProject folder:", BASE)
for f in sorted(os.listdir(BASE)):
    print("  ", f)

Created new config.json

Project folder: /content/drive/MyDrive/BETH_Triage
   00_raw_zips
   01_data_raw
   02_data_processed
   03_models
   04_scores
   05_results
   06_figures
   07_logs
   08_triage_output
   P0_setup.ipynb
   config.json


In [4]:
# ===== CELL 3: Unzip the dataset and verify it is the genuine, unchanged BETH files =====
import os, zipfile, hashlib, json, datetime

BASE = os.environ.get("BETH_BASE", "/content/drive/MyDrive/BETH_Triage")
ZIP_DIR = os.path.join(BASE, "00_raw_zips")
RAW_DIR = os.path.join(BASE, "01_data_raw")

# Fingerprints (SHA-256) and row counts of the official files.
EXPECTED = {
    "labelled_training_data.csv":   ("18ceb80a7eb4b71296df13e3967e074a81ed5b80c2212b044c061ac81a1797e2", 763144),
    "labelled_validation_data.csv": ("5a08ec47060af1cb8bb24ba632b5db0c2b1ac783aedc72693247b87c1170192e", 188967),
    "labelled_testing_data.csv":    ("e56aad64df13b9dde7be2d333f1f6b7da7f41596cb9b4105deb0fd051900706b", 188967),
}

def sha256_of(path):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def count_rows(path):
    with open(path, "rb") as fh:
        return sum(1 for _ in fh) - 1   # minus the header line

# 1) Unzip every zip in 00_raw_zips (only if the CSV is not there yet)
zips = [z for z in os.listdir(ZIP_DIR) if z.lower().endswith(".zip")]
print("Zip files found:", zips)
for z in zips:
    with zipfile.ZipFile(os.path.join(ZIP_DIR, z)) as zf:
        for name in zf.namelist():
            target = os.path.join(RAW_DIR, os.path.basename(name))
            if os.path.exists(target):
                print(f"  {os.path.basename(name)} already unzipped - skipped")
            else:
                print(f"  Unzipping {name} ...")
                with zf.open(name) as src, open(target, "wb") as dst:
                    while True:
                        buf = src.read(1024 * 1024)
                        if not buf:
                            break
                        dst.write(buf)

# 2) Verify fingerprint and row count of each CSV
manifest = {"checked_at": datetime.datetime.now().isoformat(timespec="seconds"), "files": {}}
all_ok = True
for fname, (exp_hash, exp_rows) in EXPECTED.items():
    path = os.path.join(RAW_DIR, fname)
    if not os.path.exists(path):
        print(f"MISSING: {fname}")
        all_ok = False
        continue
    got_hash = sha256_of(path)
    got_rows = count_rows(path)
    ok = (got_hash == exp_hash) and (got_rows == exp_rows)
    all_ok &= ok
    print(f"{'OK  ' if ok else 'FAIL'} {fname}: rows={got_rows:,} (expected {exp_rows:,}), sha256 match={got_hash == exp_hash}")
    manifest["files"][fname] = {"sha256": got_hash, "rows": got_rows,
                                "size_bytes": os.path.getsize(path), "verified": ok}

with open(os.path.join(BASE, "07_logs", "data_manifest.json"), "w") as fh:
    json.dump(manifest, fh, indent=2)

print("\nALL FILES VERIFIED" if all_ok else "\nSTOP: at least one file failed - see above")

Zip files found: ['labelled_validation_data.csv.zip', 'labelled_testing_data.csv.zip', 'labelled_training_data.csv.zip']
  Unzipping labelled_validation_data.csv ...
  Unzipping labelled_testing_data.csv ...
  Unzipping labelled_training_data.csv ...
OK   labelled_training_data.csv: rows=763,144 (expected 763,144), sha256 match=True
OK   labelled_validation_data.csv: rows=188,967 (expected 188,967), sha256 match=True
OK   labelled_testing_data.csv: rows=188,967 (expected 188,967), sha256 match=True

ALL FILES VERIFIED


In [5]:
# ===== CELL 4: Record the software environment (for reproducibility) =====
import os, sys, json, platform, datetime, subprocess
import numpy, pandas, sklearn

BASE = os.environ.get("BETH_BASE", "/content/drive/MyDrive/BETH_Triage")
env = {
    "recorded_at": datetime.datetime.now().isoformat(timespec="seconds"),
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "numpy": numpy.__version__,
    "pandas": pandas.__version__,
    "scikit-learn": sklearn.__version__,
}
try:
    import torch
    env["torch"] = torch.__version__
    env["cuda_available"] = torch.cuda.is_available()
except ImportError:
    env["torch"] = "not installed"
try:
    env["ram_gb"] = round(os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1e9, 1)
except (ValueError, OSError, AttributeError):
    env["ram_gb"] = "unknown"

with open(os.path.join(BASE, "07_logs", "environment.json"), "w") as fh:
    json.dump(env, fh, indent=2)
freeze = subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True).stdout
with open(os.path.join(BASE, "07_logs", "requirements_frozen.txt"), "w") as fh:
    fh.write(freeze)

for k, v in env.items():
    print(f"{k:15s} {v}")
print("\nSaved 07_logs/environment.json and 07_logs/requirements_frozen.txt")

recorded_at     2026-10-19T04:55:58
python          3.13.16
platform        Linux-6.6.122+-x86_64-with-glibc2.39
numpy           2.1.3
pandas          2.2.3
scikit-learn    1.6.1
torch           2.11.0+cpu
cuda_available  False
ram_gb          13.6

Saved 07_logs/environment.json and 07_logs/requirements_frozen.txt


In [6]:
# ===== CELL 5: Record two methodological decisions in config.json =====
import os, json, datetime

BASE = os.environ.get("BETH_BASE", "/content/drive/MyDrive/BETH_Triage")
CONFIG_PATH = os.path.join(BASE, "config.json")
with open(CONFIG_PATH) as fh:
    config = json.load(fh)

new_decisions = [
    {
        "id": "D1",
        "decision": "Primary positive label = evil; sus reported as secondary.",
        "reason": "evil marks the attacker's actions, which is the evidence investigators need. "
                  "sus also marks unusual but harmless events and appears in the training (1,269) "
                  "and validation (786) splits treated as normal.",
        "deviation_from_proposal": "Proposal section 3.4 used sus as primary.",
    },
    {
        "id": "D2",
        "decision": "Primary evaluation uses only unmodified BETH records. Prevalence-independent "
                    "metrics (false-retention rate, recall, AUROC) are measured directly; "
                    "queue precision at 1%, 5% and 10% attack share is computed with a formula. "
                    "Resampled real-record mixes are a secondary robustness check only.",
        "reason": "The test file is about 84% attack, so precision and PR-AUC on it are inflated. "
                  "Computing precision at realistic shares avoids creating or reshaping data, "
                  "so the restriction on synthetic datasets is respected.",
        "deviation_from_proposal": "Replaces the 99% volume-reduction target and Top-K capture "
                                   "as headline measures.",
    },
]
existing_ids = {d.get("id") for d in config["decisions"]}
stamp = datetime.datetime.now().isoformat(timespec="seconds")
for d in new_decisions:
    if d["id"] in existing_ids:
        print(f"{d['id']} already recorded - skipped")
    else:
        d["recorded_at"] = stamp
        config["decisions"].append(d)
        print(f"{d['id']} recorded")

with open(CONFIG_PATH, "w") as fh:
    json.dump(config, fh, indent=2)
print(f"\nconfig.json now holds {len(config['decisions'])} decision(s).")

D1 recorded
D2 recorded

config.json now holds 2 decision(s).
